In [1]:
import pickle
from pathlib import Path
import random

In [2]:
class TimeAwareMeasurement:
    """Data class for syndrome measurements with timing information."""
    def __init__(self, path_edges, histogram, duration, latency_stats=None, code_type='513'):
        """
        :param path_edges: List of tuples representing the path, e.g., [(0,1), (1,2)]
        :param histogram: A 16-dimensional numpy array of syndrome counts for [[5,1,3]] code.
        :param duration: The duration or average latency of this measurement (seconds).
        :param latency_stats: A dictionary with 'max', 'std', 'count'.
        :param code_type: String indicating the code type, e.g., '513'.
        """
        self.path_edges = path_edges
        self.histogram = histogram
        self.duration = duration
        self.latency_stats = latency_stats
        self.code_type = code_type

In [3]:
DATA_DIR = Path('../pickle_files/syndrome_data_513/')  # <-- [[5,1,3]] syndrome data

MEASUREMENTS_FILE = DATA_DIR / 'measurements.pkl'
MEASUREMENTS_TEST_FILE = DATA_DIR / 'measurements_test.pkl'
GRAPH_FILE = DATA_DIR / 'graph.pkl'

TEST_FRAC = 0.2

In [4]:
with open(MEASUREMENTS_FILE, 'rb') as f:
    measurements = pickle.load(f)

with open(GRAPH_FILE, 'rb') as f:
    graph = pickle.load(f)

edges = list(graph.edges)

m_len = len(measurements)
test_size = int(TEST_FRAC * m_len)

measurements = random.sample(measurements, m_len)  # Shuffle
all_edges = []
for idx, meas in enumerate(measurements):
    
    path = meas.path_edges
    all_edges.extend([tuple(sorted(p)) for p in path])

    if set(all_edges) >= set(edges):
        print(f"All edges covered at measurement index: {idx}")
        meas_test = measurements[idx:idx+test_size]
        break

measurements = measurements[:idx] + measurements[idx+test_size:]

print(f"Total measurements: {m_len}, Test measurements: {len(meas_test)}")
print([m.path_edges for m in meas_test])
print([m.path_edges for m in measurements if m not in meas_test])

with open(MEASUREMENTS_TEST_FILE, 'wb') as f:
    pickle.dump(meas_test, f)

with open(MEASUREMENTS_FILE, 'wb') as f:
    pickle.dump(measurements, f)

All edges covered at measurement index: 6
Total measurements: 24, Test measurements: 4
[[(3, 0), (0, 1), (1, 4)], [(0, 1), (1, 4)], [(1, 4), (4, 5), (5, 2)], [(1, 2), (2, 5)]]
[[(2, 5), (5, 4)], [(2, 5), (5, 4), (4, 3)], [(0, 1), (1, 2), (2, 5)], [(1, 4), (4, 3)], [(0, 1), (1, 4), (4, 5)], [(1, 4), (4, 5)], [(1, 0), (0, 3), (3, 4)], [(1, 2), (2, 5), (5, 4)], [(2, 1), (1, 4)], [(0, 3), (3, 4), (4, 5)], [(0, 3), (3, 4), (4, 1)], [(2, 1), (1, 4), (4, 5)], [(0, 1), (1, 4), (4, 3)], [(3, 4), (4, 5)], [(2, 1), (1, 4), (4, 3)], [(4, 1), (1, 2), (2, 5)], [(1, 0), (0, 3)], [(0, 1), (1, 2)], [(2, 1), (1, 0), (0, 3)], [(0, 3), (3, 4)]]
